In [ ]:
# 1. Install dependencies
!pip -q install pandas openpyxl numpy

In [ ]:
# 2. Mount Drive and imports

from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount message:', e)

import os
import re
import glob
import json
import hashlib
import math
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 180)

In [ ]:
# 3. Configuration

PHASEWISE_ROOT = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise"

DIRECT_BASELINE_ROOT = os.path.join(
    PHASEWISE_ROOT,
    "direct_uml_image_to_test_baseline_5reports_multi_vlm"
)

PHASE1_PART2_ROOT = os.path.join(
    PHASEWISE_ROOT,
    "phase1_part2_runs"
)

PHASE1_PART3_ROOT = os.path.join(
    PHASEWISE_ROOT,
    "phase1_part3_runs"
)

RUN_ID = "run_1"

SELECTED_REPORT_STEMS = [
    "Report_1_uml_pages",
    "Report_5_uml_pages",
    "Report_10_uml_pages",
    "Report_20_uml_pages",
    "Report_29_uml_pages",
]

# We exclude the Meta-Llama 3 8B text LLM generator from Part 3,
# but we DO keep Llama-3.2-11B-Vision as a VLM description source.
EXCLUDED_PART3_LLM_KEYWORDS = [
    "meta-llama-3-8b",
    "llama-3-8b",
    "llama_3_8b",
    "meta_llama_3_8b",
]

OUT_DIR = os.path.join(
    PHASEWISE_ROOT,
    "baseline_comparison_run1_READY_FOR_QUALITY_FIXED"
)
os.makedirs(OUT_DIR, exist_ok=True)

print('DIRECT_BASELINE_ROOT:', DIRECT_BASELINE_ROOT)
print('PHASE1_PART2_ROOT:', PHASE1_PART2_ROOT)
print('PHASE1_PART3_ROOT:', PHASE1_PART3_ROOT)
print('OUT_DIR:', OUT_DIR)
print('NOTE: This version fixes direct-baseline file discovery and robust .txt testcase parsing.')

In [ ]:
# 4. Folder/model classification helpers

def norm_name(x):
    x = str(x).strip().lower()
    x = x.replace("__", " ")
    x = x.replace("_", " ")
    x = x.replace("-", " ")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def classify_vlm_folder(folder_name):
    """Return stable VLM label for any of the five VLM source folders."""
    n = norm_name(folder_name)

    if "gemma" in n and "3" in n and "4b" in n:
        return "Gemma-3-4B"
    if "gemma" in n and "3" in n and "12b" in n:
        return "Gemma-3-12B"
    if "llava" in n or "vicuna 13b" in n:
        return "LLaVA-1.6-13B"
    if "qwen" in n and ("vl" in n or "vision" in n or "7b" in n):
        return "Qwen2.5-VL-7B"
    if "llama" in n and ("vision" in n or "11b" in n or "3 2" in n or "3.2" in str(folder_name).lower()):
        return "Llama-3.2-11B-Vision"

    return None


def classify_llm_folder(folder_name):
    """Return stable LLM generator label for the four Part 3 LLMs used in this comparison."""
    raw = str(folder_name).lower()
    n = norm_name(folder_name)

    for key in EXCLUDED_PART3_LLM_KEYWORDS:
        if key in raw.replace("_", "-") or norm_name(key) in n:
            return None

    # Qwen text LLMs
    if "qwen" in n and "14b" in n:
        return "Qwen2.5-14B-Instruct"
    if "qwen" in n and "7b" in n:
        return "Qwen2.5-7B-Instruct"

    # Mistral / Ministral. Keep BF16 in the label when the folder contains it.
    if "ministral" in n and "14b" in n:
        if "bf16" in n:
            return "Ministral-3-14B-Instruct-2512-BF16"
        return "Ministral-3-14B-Instruct-2512"
    if "mistral" in n and "7b" in n:
        return "Mistral-7B-Instruct-v0.3"

    return None


def list_subdirs(path):
    if not os.path.isdir(path):
        return []
    return [d for d in sorted(os.listdir(path)) if os.path.isdir(os.path.join(path, d))]


def is_bad_file_for_metrics(path):
    b = os.path.basename(str(path)).lower()
    bad_terms = [
        "token", "emission", "summary", "inventory", "readme", "manifest",
        "page_outputs", "raw_page_outputs", "objective", "coverage"
    ]
    return any(t in b for t in bad_terms)


def first_existing_path(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


def find_dirs_by_basename(root, accepted_names):
    """Find directories under root whose normalized basename matches accepted names."""
    if not os.path.isdir(root):
        return []
    accepted_norm = {norm_name(x) for x in accepted_names}
    out = []
    for d, subdirs, files in os.walk(root):
        if norm_name(os.path.basename(d)) in accepted_norm:
            out.append(d)
    return sorted(set(out))


def split_paths(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    return [x.strip() for x in str(value).split(';') if x.strip()]


def path_contains_run1(path):
    parts = [p.lower() for p in str(path).replace('\\', '/').split('/')]
    return RUN_ID.lower() in parts


def infer_vlm_label_from_path(path):
    """Infer VLM label from any path segment."""
    for part in str(path).replace('\\', '/').split('/'):
        lab = classify_vlm_folder(part)
        if lab:
            return lab
    return None


def infer_llm_label_from_path(path):
    """Infer LLM label from any path segment."""
    for part in str(path).replace('\\', '/').split('/'):
        lab = classify_llm_folder(part)
        if lab:
            return lab
    return None

In [ ]:
# 5. Description-file detection for REPORTS_SPLIT/reports_txt

REPORT_TXT_DIR_NAMES = [
    "reports_txt", "report_txt", "reports text", "report text", "reports-txt"
]

REPORTS_SPLIT_DIR_NAMES = [
    "REPORTS_SPLIT", "reports_split", "reports split", "REPORT_SPLIT"
]


def report_number(report_stem):
    m = re.search(r"Report[_\s-]*(\d+)", report_stem, flags=re.IGNORECASE)
    return m.group(1) if m else None


def likely_report_txt_file(path, report_stem):
    """Strict but robust check for Report_X_uml_pages.txt."""
    if not str(path).lower().endswith('.txt'):
        return False
    if is_bad_file_for_metrics(path):
        return False

    base = os.path.basename(path)
    base_no_ext = os.path.splitext(base)[0]
    n_base = norm_name(base_no_ext)
    n_report = norm_name(report_stem)

    # exact normalized match: Report_1_uml_pages.txt
    if n_base == n_report:
        return True

    # report number + uml + pages pattern
    rn = report_number(report_stem)
    if rn:
        if re.search(rf"\breport\s*{rn}\b", n_base) and "uml" in n_base and "page" in n_base:
            return True

    return False


def find_part2_run_path(vlm_folder):
    direct = os.path.join(PHASE1_PART2_ROOT, vlm_folder, RUN_ID)
    if os.path.isdir(direct):
        return direct

    # fallback: any run_1 under the VLM folder
    vlm_root = os.path.join(PHASE1_PART2_ROOT, vlm_folder)
    candidates = glob.glob(os.path.join(vlm_root, '**', RUN_ID), recursive=True)
    candidates = [c for c in candidates if os.path.isdir(c)]
    return sorted(candidates)[0] if candidates else direct


def find_part2_description_file(vlm_folder, report_stem, debug=False):
    """
    Finds the actual structured UML description file generated in Part 2.

    Main expected structure:
      phase1_part2_runs/<VLM>/run_1/REPORTS_SPLIT/reports_txt/Report_X_uml_pages.txt

    Fallbacks handle case variations and combined text files.
    """
    run_path = find_part2_run_path(vlm_folder)
    if not os.path.isdir(run_path):
        return None, run_path, []

    checked = []

    # 1) Preferred exact location: REPORTS_SPLIT/reports_txt
    preferred_dirs = []
    for rs in REPORTS_SPLIT_DIR_NAMES:
        for rt in REPORT_TXT_DIR_NAMES:
            preferred_dirs.append(os.path.join(run_path, rs, rt))

    # 2) Any reports_txt directory under run_1
    for d in find_dirs_by_basename(run_path, REPORT_TXT_DIR_NAMES):
        preferred_dirs.append(d)

    # 3) Any REPORTS_SPLIT directory itself, in case txt files are placed directly inside it
    for d in find_dirs_by_basename(run_path, REPORTS_SPLIT_DIR_NAMES):
        preferred_dirs.append(d)

    preferred_dirs = [d for d in sorted(set(preferred_dirs)) if os.path.isdir(d)]

    # Search preferred folders first
    for d in preferred_dirs:
        patterns = [
            os.path.join(d, f"{report_stem}.txt"),
            os.path.join(d, f"{report_stem.replace('_', ' ')}.txt"),
            os.path.join(d, f"{report_stem.replace('_', '-')}.txt"),
            os.path.join(d, "*.txt"),
        ]
        for pat in patterns:
            for p in glob.glob(pat):
                checked.append(p)
                if likely_report_txt_file(p, report_stem):
                    return p, run_path, checked

    # 4) General fallback under run_path
    for p in glob.glob(os.path.join(run_path, '**', '*.txt'), recursive=True):
        checked.append(p)
        if likely_report_txt_file(p, report_stem):
            return p, run_path, checked

    # 5) Combined description file fallback. We return the combined file;
    # read_description_for_report() will extract the correct report block later.
    combined_candidates = []
    combined_terms = [
        'all_uml_reports_descriptions',
        'allumlreportsdescriptions',
        'uml_reports_descriptions',
        'reports_descriptions',
        'all_reports_description',
    ]
    for p in glob.glob(os.path.join(run_path, '**', '*.txt'), recursive=True):
        b = norm_name(os.path.basename(p))
        if any(norm_name(t) in b for t in combined_terms):
            combined_candidates.append(p)

    if combined_candidates:
        return sorted(combined_candidates)[0], run_path, checked

    if debug:
        print('Description not found for:', vlm_folder, report_stem)
        print('run_path:', run_path)
        print('preferred_dirs:', preferred_dirs)
        print('checked sample:', checked[:20])

    return None, run_path, checked


def read_description_for_report(description_file, report_stem):
    """Read individual report txt or extract one report block from a combined description file."""
    if description_file is None or not os.path.exists(description_file):
        return ""

    with open(description_file, 'r', encoding='utf-8', errors='replace') as f:
        txt = f.read()

    # If this is already the individual report file, return full text.
    base = os.path.basename(description_file)
    if likely_report_txt_file(description_file, report_stem):
        return txt

    # Combined file fallback: look for REPORT: Report_X_uml_pages.pdf
    report_pdf = f"{report_stem}.pdf"
    pattern = re.compile(
        r"(REPORT\s*:\s*" + re.escape(report_pdf) + r".*?)(?=\n\s*REPORT\s*:|\Z)",
        flags=re.DOTALL | re.IGNORECASE
    )
    m = pattern.search(txt)
    if m:
        return m.group(1)

    # More permissive fallback using first occurrence of report name
    idx = txt.lower().find(report_stem.lower())
    if idx >= 0:
        next_report = re.search(r"\n\s*REPORT\s*:", txt[idx + len(report_stem):], flags=re.IGNORECASE)
        if next_report:
            end = idx + len(report_stem) + next_report.start()
            return txt[idx:end]
        return txt[idx:]

    return txt

In [ ]:
# 6. Testcase-file detection for Direct baseline and Part 3
# READY FIX: direct baseline is discovered by recursive file search, not only by one assumed folder layout.


TESTCASE_EXT_PRIORITY = {'.xlsx': 0, '.csv': 1, '.txt': 2}


def score_testcase_candidate(path, report_stem, prefer_direct=False):
    b = os.path.basename(path).lower()
    ext = os.path.splitext(b)[1].lower()
    score = 100

    score += TESTCASE_EXT_PRIORITY.get(ext, 9) * 10

    if report_stem.lower() in b:
        score -= 25
    if 'testcases' in b or 'testcase' in b or 'test cases' in b:
        score -= 20
    if prefer_direct and ('direct_baseline_testcases' in b or 'direct' in b):
        score -= 30
    if b.endswith('_testcases.xlsx') or b.endswith('_testcases.csv'):
        score -= 10
    if 'raw' in b:
        score += 20  # raw is fallback after parsed csv/xlsx
    if 'page_' in b:
        score += 50  # page-level raw files are not preferred
    if b.startswith('all_'):
        score += 30  # report-specific files are preferred
    return score


def looks_like_testcase_file(path, report_stem=None, prefer_direct=False):
    b = os.path.basename(str(path)).lower()
    if is_bad_file_for_metrics(path):
        return False
    if not b.endswith(('.xlsx', '.csv', '.txt')):
        return False
    if not ('testcase' in b or 'testcases' in b or 'test case' in b or prefer_direct):
        return False
    if report_stem:
        rn = report_number(report_stem)
        b_norm = norm_name(b)
        r_norm = norm_name(report_stem)
        parent = os.path.basename(os.path.dirname(str(path)))
        if r_norm in b_norm:
            return True
        if norm_name(parent) == r_norm:
            return True
        if rn and re.search(rf"\breport\s*{rn}\b", b_norm):
            return True
        return False
    return True


def find_testcase_files_under(search_root, report_stem, prefer_direct=False):
    if not os.path.isdir(search_root):
        return []

    all_files = []
    for ext in ['*.xlsx', '*.csv', '*.txt']:
        all_files.extend(glob.glob(os.path.join(search_root, '**', ext), recursive=True))

    candidates = [p for p in all_files if looks_like_testcase_file(p, report_stem, prefer_direct)]
    candidates = sorted(set(candidates), key=lambda p: (score_testcase_candidate(p, report_stem, prefer_direct), p))
    return candidates


def resolve_direct_baseline_roots():
    """
    Return possible direct-baseline roots. The first is the configured root.
    Fallbacks search PHASEWISE_ROOT for folders/files that clearly belong to
    the direct UML-image baseline. This fixes cases where the Drive folder name
    is slightly different or nested one level deeper.
    """
    roots = []
    if os.path.isdir(DIRECT_BASELINE_ROOT):
        roots.append(DIRECT_BASELINE_ROOT)

    # Fallback 1: folders whose basename includes direct + baseline.
    if os.path.isdir(PHASEWISE_ROOT):
        for d, subdirs, files in os.walk(PHASEWISE_ROOT):
            bn = norm_name(os.path.basename(d))
            if 'direct' in bn and 'baseline' in bn and 'page image' not in bn and 'page images' not in bn and os.path.isdir(d):
                roots.append(d)

    # Fallback 2: go upward from files named *_direct_baseline_testcases.*
    if os.path.isdir(PHASEWISE_ROOT):
        pats = [
            os.path.join(PHASEWISE_ROOT, '**', '*direct_baseline_testcases.xlsx'),
            os.path.join(PHASEWISE_ROOT, '**', '*direct_baseline_testcases.csv'),
            os.path.join(PHASEWISE_ROOT, '**', '*direct_baseline_testcases*.txt'),
        ]
        for pat in pats:
            for f in glob.glob(pat, recursive=True):
                parts = str(f).replace('\\', '/').split('/')
                # choose the ancestor just above the VLM folders if possible
                for i, part in enumerate(parts):
                    if classify_vlm_folder(part):
                        candidate_root = '/'.join(parts[:i])
                        if os.path.isdir(candidate_root):
                            roots.append(candidate_root)
                        break

    # remove nested duplicates while preserving useful roots
    out = []
    for r in sorted(set(roots), key=lambda x: (len(x), x)):
        if r not in out:
            out.append(r)
    return out


def find_direct_run_path(vlm_folder):
    # Try exact configured layout first.
    for root in resolve_direct_baseline_roots():
        direct = os.path.join(root, vlm_folder, RUN_ID)
        if os.path.isdir(direct):
            return direct
        vlm_root = os.path.join(root, vlm_folder)
        candidates = glob.glob(os.path.join(vlm_root, '**', RUN_ID), recursive=True)
        candidates = [c for c in candidates if os.path.isdir(c)]
        if candidates:
            return sorted(candidates)[0]
    return os.path.join(DIRECT_BASELINE_ROOT, vlm_folder, RUN_ID)


def recursive_direct_candidates(vlm_folder, report_stem):
    """Find direct-baseline testcase files anywhere under PHASEWISE_ROOT."""
    target_label = classify_vlm_folder(vlm_folder)
    candidates = []

    roots = resolve_direct_baseline_roots()
    if not roots and os.path.isdir(PHASEWISE_ROOT):
        roots = [PHASEWISE_ROOT]

    for root in roots:
        for ext in ['*.xlsx', '*.csv', '*.txt']:
            # Search broadly, but then filter strongly.
            for p in glob.glob(os.path.join(root, '**', ext), recursive=True):
                p_lower = p.lower()
                if 'direct' not in p_lower and 'baseline' not in p_lower:
                    continue
                if RUN_ID.lower() not in p_lower.replace('\\', '/').split('/'):
                    continue
                if target_label and infer_vlm_label_from_path(p) != target_label:
                    continue
                if looks_like_testcase_file(p, report_stem, prefer_direct=True):
                    candidates.append(p)

    return sorted(set(candidates), key=lambda p: (score_testcase_candidate(p, report_stem, True), p))


def find_direct_testcase_file(vlm_folder, report_stem):
    run_path = find_direct_run_path(vlm_folder)
    candidates = []

    # Preferred expected output folder.
    if os.path.isdir(run_path):
        output_roots = find_dirs_by_basename(run_path, ['DIRECT_UML_IMAGE_TO_TESTCASES'])
        if not output_roots:
            output_roots = [run_path]
        for root in output_roots:
            candidates.extend(find_testcase_files_under(root, report_stem, prefer_direct=True))

    # Robust fallback: search by file name and path context.
    candidates.extend(recursive_direct_candidates(vlm_folder, report_stem))

    candidates = sorted(set(candidates), key=lambda p: (score_testcase_candidate(p, report_stem, True), p))
    selected = candidates[0] if candidates else None
    return selected, run_path, candidates


def find_direct_token_csvs(vlm_folder):
    """Best-effort direct-token CSV discovery for the given VLM/run_1."""
    run_path = find_direct_run_path(vlm_folder)
    csvs = find_token_csvs(run_path) if os.path.isdir(run_path) else []

    target_label = classify_vlm_folder(vlm_folder)
    for root in resolve_direct_baseline_roots():
        for p in glob.glob(os.path.join(root, '**', '*.csv'), recursive=True):
            b = os.path.basename(p).lower()
            if 'token' not in b:
                continue
            if RUN_ID.lower() not in p.lower().replace('\\', '/').split('/'):
                continue
            if target_label and infer_vlm_label_from_path(p) != target_label:
                continue
            csvs.append(p)
    return sorted(set(csvs))


def find_part3_run_path(llm_folder, vlm_folder):
    direct = os.path.join(PHASE1_PART3_ROOT, llm_folder, vlm_folder, RUN_ID)
    if os.path.isdir(direct):
        return direct
    combo_root = os.path.join(PHASE1_PART3_ROOT, llm_folder, vlm_folder)
    candidates = glob.glob(os.path.join(combo_root, '**', RUN_ID), recursive=True)
    candidates = [c for c in candidates if os.path.isdir(c)]
    return sorted(candidates)[0] if candidates else direct


def find_part3_testcase_file(llm_folder, vlm_folder, report_stem):
    run_path = find_part3_run_path(llm_folder, vlm_folder)
    if not os.path.isdir(run_path):
        return None, run_path, []

    output_roots = find_dirs_by_basename(run_path, ['TESTCASES_LLM_ALL', 'TESTCASES LLM ALL', 'testcases_llm_all'])
    if not output_roots:
        output_roots = [run_path]

    candidates = []
    for root in output_roots:
        candidates.extend(find_testcase_files_under(root, report_stem, prefer_direct=False))

    candidates = sorted(set(candidates), key=lambda p: (score_testcase_candidate(p, report_stem, False), p))
    selected = candidates[0] if candidates else None
    return selected, run_path, candidates


def find_token_csvs(root):
    if root is None or not os.path.isdir(root):
        return []
    csvs = glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)
    out = []
    for p in csvs:
        b = os.path.basename(p).lower()
        if 'token' in b:
            out.append(p)
    return sorted(set(out))

print('Resolved direct-baseline roots:')
for r in resolve_direct_baseline_roots():
    print('  ', r)

In [ ]:
# 7. Build corrected inventory

# Discover actual VLM folders from Direct + Part 2 roots.
vlm_folder_map = {}
for root in [DIRECT_BASELINE_ROOT, PHASE1_PART2_ROOT]:
    for folder in list_subdirs(root):
        label = classify_vlm_folder(folder)
        if label:
            vlm_folder_map[label] = folder

print('Detected VLM folders:')
for k, v in sorted(vlm_folder_map.items()):
    print(f'  {k:25s} -> {v}')

# Discover Part 3 LLM folders and VLM child folders.
llm_folder_map = {}
for llm_folder in list_subdirs(PHASE1_PART3_ROOT):
    label = classify_llm_folder(llm_folder)
    if label:
        llm_folder_map[label] = llm_folder

print('\nDetected Part 3 LLM folders:')
for k, v in sorted(llm_folder_map.items()):
    print(f'  {k:35s} -> {v}')

manifest_rows = []
direct_inventory_rows = []
part2_description_rows = []
part3_inventory_rows = []

# Part 2 description inventory for all VLM/report pairs.
description_lookup = {}
for vlm_label, vlm_folder in sorted(vlm_folder_map.items()):
    for report in SELECTED_REPORT_STEMS:
        desc_file, p2_run_path, checked = find_part2_description_file(vlm_folder, report)
        desc_text = read_description_for_report(desc_file, report) if desc_file else ''
        ok = bool(desc_file and desc_text.strip())
        description_lookup[(vlm_folder, report)] = desc_file
        part2_description_rows.append({
            'vlm_label': vlm_label,
            'vlm_folder': vlm_folder,
            'report': report,
            'part2_run_path': p2_run_path,
            'selected_description_file': desc_file,
            'description_found': int(ok),
            'description_text_chars': len(desc_text),
            'checked_txt_count': len(checked),
            'checked_txt_sample': '; '.join(checked[:8]),
            'part2_token_csvs': '; '.join(find_token_csvs(p2_run_path)),
            'status': 'ok' if ok else 'missing_description_file'
        })

# Direct baseline inventory and manifest rows.
for vlm_label, vlm_folder in sorted(vlm_folder_map.items()):
    direct_run_path = find_direct_run_path(vlm_folder)
    direct_token_csvs = find_direct_token_csvs(vlm_folder)

    for report in SELECTED_REPORT_STEMS:
        testcase_file, run_path, candidates = find_direct_testcase_file(vlm_folder, report)
        desc_file = description_lookup.get((vlm_folder, report))

        row = {
            'comparison_group': 'direct_baseline',
            'approach': 'Direct UML image -> VLM test cases',
            'vlm_label': vlm_label,
            'vlm_folder': vlm_folder,
            'llm_generator': 'N/A',
            'llm_folder': 'N/A',
            'report': report,
            'run_id': RUN_ID,
            'testcase_file': testcase_file,
            'testcase_status': 'ok' if testcase_file else 'missing_testcase_file',
            'description_file_for_structural_reference': desc_file,
            'description_status': 'ok' if desc_file else 'missing_description_reference',
            'direct_token_csvs': '; '.join(direct_token_csvs),
            'part2_token_csvs': '; '.join(find_token_csvs(find_part2_run_path(vlm_folder))),
            'part3_token_csvs': '',
            'candidate_testcase_files': '; '.join(candidates[:10]),
            'ready_for_quality_metrics': int(bool(testcase_file)),
            'ready_for_structural_metrics': int(bool(testcase_file and desc_file)),
        }
        manifest_rows.append(row)
        direct_inventory_rows.append(row.copy())

# Part 3 cross-product inventory and manifest rows.
for llm_label, llm_folder in sorted(llm_folder_map.items()):
    llm_root = os.path.join(PHASE1_PART3_ROOT, llm_folder)

    # Use actual VLM subfolders in each LLM directory, not a forced one-to-one mapping.
    for part3_vlm_folder in list_subdirs(llm_root):
        vlm_label = classify_vlm_folder(part3_vlm_folder)
        if not vlm_label:
            continue

        # For description reference, use the matching Part 2 VLM folder.
        # Usually it has the same folder name, but we map by label if needed.
        part2_vlm_folder = vlm_folder_map.get(vlm_label, part3_vlm_folder)

        p3_run_path = find_part3_run_path(llm_folder, part3_vlm_folder)
        p3_token_csvs = find_token_csvs(p3_run_path)
        p2_token_csvs = find_token_csvs(find_part2_run_path(part2_vlm_folder))

        for report in SELECTED_REPORT_STEMS:
            testcase_file, run_path, candidates = find_part3_testcase_file(llm_folder, part3_vlm_folder, report)
            desc_file = description_lookup.get((part2_vlm_folder, report))
            if desc_file is None:
                desc_file, _, _ = find_part2_description_file(part2_vlm_folder, report)

            row = {
                'comparison_group': 'two_stage_pipeline_cross_product',
                'approach': 'Two-stage UML image -> VLM description -> LLM test cases',
                'vlm_label': vlm_label,
                'vlm_folder': part3_vlm_folder,
                'part2_vlm_folder_for_description': part2_vlm_folder,
                'llm_generator': llm_label,
                'llm_folder': llm_folder,
                'report': report,
                'run_id': RUN_ID,
                'testcase_file': testcase_file,
                'testcase_status': 'ok' if testcase_file else 'missing_testcase_file',
                'description_file_for_structural_reference': desc_file,
                'description_status': 'ok' if desc_file else 'missing_description_reference',
                'direct_token_csvs': '',
                'part2_token_csvs': '; '.join(p2_token_csvs),
                'part3_token_csvs': '; '.join(p3_token_csvs),
                'candidate_testcase_files': '; '.join(candidates[:10]),
                'ready_for_quality_metrics': int(bool(testcase_file)),
                'ready_for_structural_metrics': int(bool(testcase_file and desc_file)),
            }
            manifest_rows.append(row)
            part3_inventory_rows.append(row.copy())

manifest_df = pd.DataFrame(manifest_rows)
direct_inventory_df = pd.DataFrame(direct_inventory_rows)
part2_description_df = pd.DataFrame(part2_description_rows)
part3_inventory_df = pd.DataFrame(part3_inventory_rows)

print('\nManifest readiness:')
display(manifest_df.groupby(['comparison_group']).agg(
    rows=('report','count'),
    testcases_ready=('ready_for_quality_metrics','sum'),
    structural_ready=('ready_for_structural_metrics','sum'),
    vlms=('vlm_label','nunique'),
    llms=('llm_generator','nunique'),
    reports=('report','nunique')
).reset_index())

print('\nPart 2 description readiness:')
display(part2_description_df.groupby(['vlm_label']).agg(
    reports=('report','count'),
    descriptions_found=('description_found','sum'),
    avg_chars=('description_text_chars','mean')
).reset_index())

# Save inventory immediately.
manifest_df.to_csv(os.path.join(OUT_DIR, 'comparison_input_manifest_run1_DESCRIPTION_FIXED.csv'), index=False)
direct_inventory_df.to_csv(os.path.join(OUT_DIR, 'direct_baseline_inventory_run1_DESCRIPTION_FIXED.csv'), index=False)
part2_description_df.to_csv(os.path.join(OUT_DIR, 'phase1_part2_description_inventory_run1_DESCRIPTION_FIXED.csv'), index=False)
part3_inventory_df.to_csv(os.path.join(OUT_DIR, 'phase1_part3_crossproduct_inventory_run1_DESCRIPTION_FIXED.csv'), index=False)

with pd.ExcelWriter(os.path.join(OUT_DIR, 'inventory_run1_DESCRIPTION_FIXED.xlsx'), engine='openpyxl') as writer:
    manifest_df.to_excel(writer, sheet_name='manifest', index=False)
    direct_inventory_df.to_excel(writer, sheet_name='direct_inventory', index=False)
    part2_description_df.to_excel(writer, sheet_name='part2_descriptions', index=False)
    part3_inventory_df.to_excel(writer, sheet_name='part3_crossproduct', index=False)

print('Inventory saved to:', OUT_DIR)

In [ ]:
# 8. Testcase parsing helpers for xlsx/csv/txt

REQUIRED_COLUMNS = [
    'Test Case ID',
    'Title',
    'Module',
    'Source UML Pages',
    'Preconditions',
    'Test Data',
    'Test Steps',
    'Expected Result',
]

FIELD_ALIASES = {
    'Test Case ID': ['Test Case ID', 'TestCase ID', 'Test ID', 'ID', 'TC ID', 'Case ID'],
    'Title': ['Title', 'Test Case Title', 'Test Title', 'Scenario'],
    'Module': ['Module', 'Feature', 'Component', 'Class', 'Functionality'],
    'Source UML Pages': ['Source UML Pages', 'Source Pages', 'UML Pages', 'Source UML Page', 'Page', 'Pages'],
    'Preconditions': ['Preconditions', 'Precondition', 'Pre Conditions', 'Prerequisites'],
    'Test Data': ['Test Data', 'Data', 'Input Data', 'Inputs'],
    'Test Steps': ['Test Steps', 'Steps', 'Procedure', 'Actions'],
    'Expected Result': ['Expected Result', 'Expected Results', 'Oracle', 'Expected Output', 'Expected Outcome'],
}

# Flatten aliases once for regex.
ALL_FIELD_ALIASES = []
for aliases in FIELD_ALIASES.values():
    ALL_FIELD_ALIASES.extend(aliases)
FIELD_PATTERN = '|'.join(sorted((re.escape(a) for a in ALL_FIELD_ALIASES), key=len, reverse=True))


def clean_text(x):
    if pd.isna(x):
        return ''
    x = str(x)
    x = x.replace('\r', ' ').strip()
    return re.sub(r'\s+', ' ', x).strip()


def normalize_colname(c):
    c = str(c).strip().lower()
    c = re.sub(r'[^a-z0-9]+', ' ', c)
    return re.sub(r'\s+', ' ', c).strip()


def standardize_columns(df):
    df = df.copy()
    norm_to_orig = {normalize_colname(c): c for c in df.columns}
    rename = {}
    for standard, aliases in FIELD_ALIASES.items():
        for alias in aliases:
            key = normalize_colname(alias)
            if key in norm_to_orig:
                rename[norm_to_orig[key]] = standard
                break
    df = df.rename(columns=rename)
    for c in REQUIRED_COLUMNS:
        if c not in df.columns:
            df[c] = ''
    return df[REQUIRED_COLUMNS + [c for c in df.columns if c not in REQUIRED_COLUMNS]]


def markdown_label_regex(alias):
    # Allows: Test Case ID:, **Test Case ID:**, - **Test Case ID:**, 1. Test Case ID -
    alias_pat = re.escape(alias).replace(r'\ ', r'\s+')
    return (
        r'(?im)^\s*(?:[-*•]\s*)?(?:\d+[\.)]\s*)?'
        r'(?:\*\*)?\s*' + alias_pat + r'\s*(?:\*\*)?\s*[:\-–—]\s*'
    )


def extract_field_from_block(block, aliases):
    for alias in aliases:
        start_re = re.compile(markdown_label_regex(alias))
        m = start_re.search(block)
        if not m:
            continue
        start = m.end()
        # next field label begins the next field
        next_re = re.compile(
            r'(?im)^\s*(?:[-*•]\s*)?(?:\d+[\.)]\s*)?(?:\*\*)?\s*(?:' + FIELD_PATTERN + r')\s*(?:\*\*)?\s*[:\-–—]\s*'
        )
        next_m = next_re.search(block, start)
        end = next_m.start() if next_m else len(block)
        return block[start:end].strip().strip('|').strip()
    return ''


def split_txt_into_testcase_blocks(text):
    # Prefer explicit Test Case ID labels.
    id_patterns = [
        r'(?im)^\s*(?:[-*•]\s*)?(?:\d+[\.)]\s*)?(?:\*\*)?\s*Test\s*Case\s*ID\s*(?:\*\*)?\s*[:\-–—]',
        r'(?im)^\s*(?:[-*•]\s*)?(?:\d+[\.)]\s*)?(?:\*\*)?\s*TC\s*ID\s*(?:\*\*)?\s*[:\-–—]',
    ]
    starts = []
    for pat in id_patterns:
        starts.extend(list(re.finditer(pat, text)))
    starts = sorted(starts, key=lambda m: m.start())

    # Fallback: headings like ### Test Case 1 or Test Case 1:
    if not starts:
        starts = list(re.finditer(r'(?im)^\s*(?:#+\s*)?(?:\*\*)?\s*Test\s*Case\s*\d+\s*(?:\*\*)?\s*[:\-–—]?\s*$', text))

    if not starts:
        return []

    blocks = []
    for i, m in enumerate(starts):
        start = m.start()
        end = starts[i+1].start() if i+1 < len(starts) else len(text)
        block = text[start:end].strip()
        if block:
            blocks.append(block)
    return blocks


def parse_markdown_table_from_txt(text):
    """Parse pipe-table test cases when the model outputs a markdown table."""
    lines = [ln.rstrip() for ln in text.splitlines()]
    table_blocks = []
    cur = []
    for ln in lines:
        if '|' in ln and re.search(r'test|title|module|step|expected|page', ln, flags=re.IGNORECASE):
            cur.append(ln)
        elif cur:
            if len(cur) >= 2:
                table_blocks.append(cur)
            cur = []
    if cur and len(cur) >= 2:
        table_blocks.append(cur)

    rows = []
    for block in table_blocks:
        # remove separator rows
        useful = [ln for ln in block if not re.match(r'^\s*\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?\s*$', ln)]
        if len(useful) < 2:
            continue
        header = [h.strip().strip('*') for h in useful[0].strip('|').split('|')]
        data_lines = useful[1:]

        # map table columns to required fields
        col_map = {}
        for idx, h in enumerate(header):
            hn = normalize_colname(h)
            for standard, aliases in FIELD_ALIASES.items():
                if any(normalize_colname(a) == hn or normalize_colname(a) in hn or hn in normalize_colname(a) for a in aliases):
                    col_map[idx] = standard
                    break

        if not col_map:
            continue

        for dl in data_lines:
            vals = [v.strip() for v in dl.strip('|').split('|')]
            if len(vals) < 2:
                continue
            row = {c: '' for c in REQUIRED_COLUMNS}
            for idx, val in enumerate(vals):
                if idx in col_map:
                    row[col_map[idx]] = val.strip()
            if any(row[c] for c in REQUIRED_COLUMNS):
                rows.append(row)

    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=REQUIRED_COLUMNS)


def parse_testcases_from_txt(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        text = f.read()

    # Remove code fences but keep inside content.
    text = re.sub(r'```(?:text|markdown|json|csv)?', '', text, flags=re.IGNORECASE)
    text = text.replace('```', '')

    # First try pipe/markdown table format.
    table_df = parse_markdown_table_from_txt(text)
    if len(table_df) > 0:
        return standardize_columns(table_df)

    # Then try labeled block format.
    blocks = split_txt_into_testcase_blocks(text)
    rows = []

    for bi, block in enumerate(blocks, start=1):
        row = {}
        for standard, aliases in FIELD_ALIASES.items():
            row[standard] = extract_field_from_block(block, aliases)

        # Fallback ID from heading if missing.
        if not row.get('Test Case ID'):
            m = re.search(r'(?im)Test\s*Case\s*(\d+)', block)
            if m:
                row['Test Case ID'] = f'TC-{int(m.group(1)):03d}'

        # Keep only non-empty real test cases.
        if any(clean_text(row.get(c, '')) for c in ['Title', 'Test Steps', 'Expected Result', 'Test Case ID']):
            rows.append(row)

    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=REQUIRED_COLUMNS)


def robust_read_testcase_file(path):
    path = str(path)
    if path.lower().endswith('.xlsx'):
        return standardize_columns(pd.read_excel(path))
    if path.lower().endswith('.csv'):
        try:
            return standardize_columns(pd.read_csv(path, encoding='utf-8-sig'))
        except Exception:
            return standardize_columns(pd.read_csv(path, encoding='utf-8-sig', engine='python'))
    if path.lower().endswith('.txt'):
        return standardize_columns(parse_testcases_from_txt(path))
    raise ValueError(f'Unsupported testcase file type: {path}')


def txt_parser_diagnostic(path, max_chars=1200):
    """Use this only when a txt file is parsed as 0 rows."""
    try:
        with open(path, 'r', encoding='utf-8', errors='replace') as f:
            t = f.read(max_chars)
        return t.replace('\n', ' ')[:max_chars]
    except Exception as e:
        return f'Cannot read sample: {e}'

In [ ]:
# 9. UML objective extraction from Phase 1 Part 2 descriptions

def normalize_text_for_match(text):
    text = '' if pd.isna(text) else str(text)
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = text.lower().replace('_', ' ')
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def parse_page_blocks_from_description(txt):
    lines = txt.splitlines()
    blocks = []
    current_page = None
    current_lines = []
    page_re = re.compile(r'page\s+(\d+)', flags=re.IGNORECASE)

    for line in lines:
        # Expected: ===== Report_1_uml_pages.pdf | Page 1 =====
        if '====' in line and re.search(r'page\s+\d+', line, flags=re.IGNORECASE):
            if current_page is not None:
                blocks.append({'page': current_page, 'text': '\n'.join(current_lines)})
            m = page_re.search(line)
            current_page = int(m.group(1)) if m else None
            current_lines = [line]
        else:
            if current_page is not None:
                current_lines.append(line)

    if current_page is not None:
        blocks.append({'page': current_page, 'text': '\n'.join(current_lines)})

    return blocks


def section_lines(block_text, section_name):
    known_sections = [
        'UML Type:', 'Components:', 'Actors / Lifelines:', 'Relationships / Links:',
        'Main Flows', 'Main Flows (step by step):', 'Conditions / Branches:',
        'Loops / Repetitions:', 'Notes / Annotations:', 'Observations:'
    ]
    lines = block_text.splitlines()
    capture = False
    out = []
    for line in lines:
        stripped = line.strip()
        if stripped.lower().startswith(section_name.lower()):
            capture = True
            continue
        if capture:
            if any(stripped.lower().startswith(s.lower()) for s in known_sections if not s.lower().startswith(section_name.lower())):
                break
            if stripped:
                out.append(stripped)
    return out


def clean_objective_line(line):
    line = str(line).strip()
    line = re.sub(r'^[\-\*\d\.\)\s]+', '', line).strip()
    return re.sub(r'\s+', ' ', line)


def extract_component_name(line):
    line = clean_objective_line(line)
    line = line.split('—')[0].strip()
    line = line.split(';')[0].strip()
    if normalize_text_for_match(line) in {'', 'none', 'none none'}:
        return ''
    if line.strip() in {'-', '*'}:
        return ''
    return line


def extract_relationship_action(line):
    line = clean_objective_line(line)
    if normalize_text_for_match(line) in {'', 'none'}:
        return ''

    # Prefer quoted labels/messages.
    for key in ['label', 'message']:
        m = re.search(rf'{key}\s*=\s*"([^"]+)"', line, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip()

    # Sequence format: A -> B: action
    if ':' in line:
        return line.split(':', 1)[-1].strip()

    return line


def meaningful_tokens(text):
    text = normalize_text_for_match(text)
    stop = {
        'the', 'a', 'an', 'and', 'or', 'to', 'from', 'with', 'of', 'in', 'on',
        'is', 'are', 'be', 'by', 'for', 'as', 'if', 'then', 'else', 'return',
        'returns', 'none', 'type', 'label', 'message', 'explanation', 'main',
        'flow', 'step', 'true', 'false', 'kind', 'page', 'uml'
    }
    return [t for t in text.split() if len(t) >= 3 and t not in stop]


def extract_uml_objectives_from_description_text(description_text):
    blocks = parse_page_blocks_from_description(description_text)
    rows = []

    for block in blocks:
        page = block['page']
        text = block['text']

        # Components and actors/lifelines.
        for line in section_lines(text, 'Components:') + section_lines(text, 'Actors / Lifelines:'):
            name = extract_component_name(line)
            if name:
                rows.append({
                    'page': page,
                    'objective_type': 'component_or_actor',
                    'objective_text': name,
                    'raw_line': line,
                })

        # Relationships/messages.
        for line in section_lines(text, 'Relationships / Links:'):
            action = extract_relationship_action(line)
            if action:
                rows.append({
                    'page': page,
                    'objective_type': 'relationship_or_message',
                    'objective_text': action,
                    'raw_line': line,
                })

        # Main flows.
        flow_lines = section_lines(text, 'Main Flows')
        if not flow_lines:
            flow_lines = section_lines(text, 'Main Flows (step by step):')
        for line in flow_lines:
            line_clean = clean_objective_line(line)
            if line_clean and normalize_text_for_match(line_clean) != 'none':
                rows.append({
                    'page': page,
                    'objective_type': 'main_flow',
                    'objective_text': line_clean,
                    'raw_line': line,
                })

        # Conditions/branches.
        for line in section_lines(text, 'Conditions / Branches:'):
            line_clean = clean_objective_line(line)
            if line_clean and normalize_text_for_match(line_clean) != 'none':
                rows.append({
                    'page': page,
                    'objective_type': 'condition_or_branch',
                    'objective_text': line_clean,
                    'raw_line': line,
                })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        df = pd.DataFrame(columns=['page', 'objective_type', 'objective_text', 'raw_line'])

    # Remove accidental duplicate objectives within the same report.
    df['_key'] = df.apply(lambda r: (r['page'], r['objective_type'], normalize_text_for_match(r['objective_text'])), axis=1)
    df = df.drop_duplicates('_key').drop(columns=['_key']).reset_index(drop=True)
    return df

In [ ]:
# 10. Output-level and structural metrics

def parse_source_pages(value, expected_pages=None):
    text = clean_text(value)
    if not text:
        return set()
    nums = re.findall(r'\d+', text)
    pages = set()
    for n in nums:
        try:
            p = int(n)
            if expected_pages is None or len(expected_pages) == 0 or p in expected_pages:
                pages.add(p)
        except Exception:
            pass
    return pages


def count_steps(text):
    text = '' if pd.isna(text) else str(text)
    matches = re.findall(r'(^|\n)\s*\d+\.', text)
    if matches:
        return len(matches)
    return 1 if clean_text(text) else 0


def normalize_full_row_for_duplicate(text):
    text = '' if text is None or pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'\btc[-_a-z0-9]*[-_]\d+\b', ' ', text)
    text = re.sub(r'\btest case id\s*:\s*\S+', ' ', text)
    text = re.sub(r'\b\d+\.\s*', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def row_hash(row):
    parts = [f'{c}: {clean_text(row.get(c, ""))}' for c in REQUIRED_COLUMNS]
    txt = '. '.join(parts)
    norm = normalize_full_row_for_duplicate(txt)
    return hashlib.md5(norm.encode('utf-8')).hexdigest()


def testcase_df_to_texts(df):
    df = standardize_columns(df)
    texts = []
    for _, row in df.iterrows():
        parts = [str(row.get(c, '')) for c in REQUIRED_COLUMNS]
        texts.append(normalize_text_for_match(' '.join(parts)))
    return texts


def objective_is_covered(objective_text, testcase_texts, objective_type):
    obj_norm = normalize_text_for_match(objective_text)
    obj_tokens = set(meaningful_tokens(objective_text))
    if not obj_norm or not testcase_texts:
        return False, ''

    # Short component names: direct containment is fine.
    if objective_type == 'component_or_actor':
        for t in testcase_texts:
            if obj_norm in t:
                return True, t[:250]
        return False, ''

    if not obj_tokens:
        return False, ''

    if objective_type == 'condition_or_branch':
        required = max(1, math.ceil(0.60 * len(obj_tokens)))
        if len(obj_tokens) >= 2:
            required = max(2, required)
    else:
        required = max(1 if len(obj_tokens) <= 2 else 2, math.ceil(0.50 * len(obj_tokens)))

    for t in testcase_texts:
        hit = sum(1 for tok in obj_tokens if tok in t)
        if hit >= required:
            return True, t[:250]
    return False, ''


def summarize_testcase_output(testcase_df, objective_df):
    n = len(testcase_df)
    expected_pages = sorted(set(objective_df['page'].dropna().astype(int).tolist())) if len(objective_df) else []

    all_pages = set()
    for v in testcase_df['Source UML Pages'].tolist():
        all_pages |= parse_source_pages(v, set(expected_pages))

    if expected_pages:
        missing_pages = sorted(set(expected_pages) - all_pages)
        page_coverage = len(all_pages) / len(expected_pages)
    else:
        missing_pages = []
        page_coverage = np.nan

    empty_counts = {f'empty_{c}': int(testcase_df[c].apply(lambda x: clean_text(x) == '').sum()) for c in REQUIRED_COLUMNS}
    total_empty = sum(empty_counts.values())

    step_counts = testcase_df['Test Steps'].apply(count_steps) if n else pd.Series(dtype=int)

    if n:
        hashes = testcase_df.apply(row_hash, axis=1)
        duplicate_count = n - hashes.nunique()
        duplicate_rate = duplicate_count / n
    else:
        duplicate_count = 0
        duplicate_rate = 0.0

    modules = sorted(set(clean_text(x) for x in testcase_df['Module'].tolist() if clean_text(x)))

    out = {
        'test_cases': n,
        'expected_page_count': len(expected_pages),
        'covered_page_count': len(all_pages),
        'covered_pages': ','.join(map(str, sorted(all_pages))),
        'missing_pages': ','.join(map(str, missing_pages)),
        'page_coverage': page_coverage,
        'avg_step_count': float(step_counts.mean()) if n else 0.0,
        'exact_duplicate_count': int(duplicate_count),
        'exact_duplicate_rate': float(duplicate_rate),
        'unique_modules_count': len(modules),
        'modules': '; '.join(modules),
        'total_empty_required_fields': int(total_empty),
    }
    out.update(empty_counts)
    return out


def compute_structural_coverage(testcase_df, objective_df):
    testcase_texts = testcase_df_to_texts(testcase_df)
    details = []

    for _, obj in objective_df.iterrows():
        covered, matched_sample = objective_is_covered(obj['objective_text'], testcase_texts, obj['objective_type'])
        details.append({
            'page': obj['page'],
            'objective_type': obj['objective_type'],
            'objective_text': obj['objective_text'],
            'raw_line': obj['raw_line'],
            'covered': int(covered),
            'matched_testcase_sample': matched_sample,
        })

    detail_df = pd.DataFrame(details)
    if len(detail_df) == 0:
        return {
            'total_objectives': 0,
            'covered_objectives': 0,
            'overall_structural_coverage': np.nan,
            'component_or_actor_coverage': np.nan,
            'relationship_or_message_coverage': np.nan,
            'main_flow_coverage': np.nan,
            'condition_or_branch_coverage': np.nan,
        }, detail_df

    summary = {
        'total_objectives': len(detail_df),
        'covered_objectives': int(detail_df['covered'].sum()),
        'overall_structural_coverage': float(detail_df['covered'].mean()),
    }

    for t in ['component_or_actor', 'relationship_or_message', 'main_flow', 'condition_or_branch']:
        sub = detail_df[detail_df['objective_type'] == t]
        summary[f'{t}_total'] = len(sub)
        summary[f'{t}_covered'] = int(sub['covered'].sum()) if len(sub) else 0
        summary[f'{t}_coverage'] = float(sub['covered'].mean()) if len(sub) else np.nan

    return summary, detail_df

In [ ]:
# 11. Token-count extraction

def safe_num(x):
    try:
        if pd.isna(x):
            return 0.0
        return float(x)
    except Exception:
        return 0.0


def infer_report_from_value(x):
    s = str(x)
    m = re.search(r'(Report[_\s-]*\d+[_\s-]*uml[_\s-]*pages)', s, flags=re.IGNORECASE)
    if m:
        return re.sub(r'\s+|-', '_', m.group(1))
    return ''


def aggregate_tokens_from_csvs(csv_list, report_stem):
    if not csv_list:
        return {'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'token_rows_found': 0}

    input_total = 0.0
    output_total = 0.0
    total_total = 0.0
    rows_found = 0

    for csv_path in csv_list:
        if not csv_path or not os.path.exists(csv_path):
            continue
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue
        if len(df) == 0:
            continue

        norm_cols = {normalize_colname(c): c for c in df.columns}

        input_cols = [c for k, c in norm_cols.items() if ('input' in k or 'prompt' in k) and 'token' in k]
        output_cols = [c for k, c in norm_cols.items() if ('output' in k or 'completion' in k or 'generated' in k) and 'token' in k]
        total_cols = [c for k, c in norm_cols.items() if 'total' in k and 'token' in k]

        # identify report rows using any column that contains report names
        mask = pd.Series([False] * len(df))
        for c in df.columns:
            if df[c].astype(str).str.contains(report_stem, case=False, na=False).any():
                mask |= df[c].astype(str).str.contains(report_stem, case=False, na=False)
            else:
                rn = report_number(report_stem)
                if rn:
                    mask |= df[c].astype(str).str.contains(rf'Report[_\s-]*{rn}[_\s-]*uml[_\s-]*pages', case=False, regex=True, na=False)

        sub = df[mask]
        if len(sub) == 0:
            continue

        rows_found += len(sub)
        if input_cols:
            input_total += sub[input_cols[0]].apply(safe_num).sum()
        if output_cols:
            output_total += sub[output_cols[0]].apply(safe_num).sum()
        if total_cols:
            total_total += sub[total_cols[0]].apply(safe_num).sum()
        else:
            total_total += (sub[input_cols[0]].apply(safe_num).sum() if input_cols else 0) + (sub[output_cols[0]].apply(safe_num).sum() if output_cols else 0)

    return {
        'input_tokens': int(input_total),
        'output_tokens': int(output_total),
        'total_tokens': int(total_total),
        'token_rows_found': int(rows_found),
    }

In [ ]:
# 12. Run structural comparison

metric_rows = []
objective_detail_tables = []
metric_errors = []

for idx, row in manifest_df.iterrows():
    if int(row.get('ready_for_quality_metrics', 0)) != 1:
        metric_errors.append({**row.to_dict(), 'metric_error': 'not_ready_for_quality_metrics'})
        continue

    testcase_file = row['testcase_file']
    desc_file = row.get('description_file_for_structural_reference', None)
    report = row['report']

    try:
        testcase_df = robust_read_testcase_file(testcase_file)
        if len(testcase_df) == 0:
            sample = txt_parser_diagnostic(testcase_file) if str(testcase_file).lower().endswith('.txt') else ''
            metric_errors.append({**row.to_dict(), 'metric_error': 'parsed_zero_testcases', 'parser_sample': sample})
    except Exception as e:
        metric_errors.append({**row.to_dict(), 'metric_error': f'testcase_parse_error: {type(e).__name__}: {e}'})
        continue

    description_text = read_description_for_report(desc_file, report) if desc_file else ''
    objective_df = extract_uml_objectives_from_description_text(description_text) if description_text.strip() else pd.DataFrame(columns=['page','objective_type','objective_text','raw_line'])

    try:
        output_summary = summarize_testcase_output(testcase_df, objective_df)
        structural_summary, detail_df = compute_structural_coverage(testcase_df, objective_df)

        # Token counts depend on approach.
        if row['comparison_group'] == 'direct_baseline':
            token_csvs = split_paths(row.get('direct_token_csvs', ''))
            tok = aggregate_tokens_from_csvs(token_csvs, report)
            part2_tok = {'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'token_rows_found': 0}
            part3_tok = {'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'token_rows_found': 0}
        else:
            p2_csvs = split_paths(row.get('part2_token_csvs', ''))
            p3_csvs = split_paths(row.get('part3_token_csvs', ''))
            part2_tok = aggregate_tokens_from_csvs(p2_csvs, report)
            part3_tok = aggregate_tokens_from_csvs(p3_csvs, report)
            tok = {
                'input_tokens': part2_tok['input_tokens'] + part3_tok['input_tokens'],
                'output_tokens': part2_tok['output_tokens'] + part3_tok['output_tokens'],
                'total_tokens': part2_tok['total_tokens'] + part3_tok['total_tokens'],
                'token_rows_found': part2_tok['token_rows_found'] + part3_tok['token_rows_found'],
            }

        metric_row = row.to_dict()
        metric_row.update(output_summary)
        metric_row.update(structural_summary)
        metric_row.update({
            'input_tokens': tok['input_tokens'],
            'output_tokens': tok['output_tokens'],
            'total_tokens': tok['total_tokens'],
            'token_rows_found': tok['token_rows_found'],
            'part2_input_tokens': part2_tok['input_tokens'],
            'part2_output_tokens': part2_tok['output_tokens'],
            'part2_total_tokens': part2_tok['total_tokens'],
            'part3_input_tokens': part3_tok['input_tokens'],
            'part3_output_tokens': part3_tok['output_tokens'],
            'part3_total_tokens': part3_tok['total_tokens'],
            'parsed_testcase_rows': len(testcase_df),
            'extracted_objective_rows': len(objective_df),
        })
        metric_rows.append(metric_row)

        if len(detail_df):
            detail_df = detail_df.copy()
            detail_df['comparison_group'] = row['comparison_group']
            detail_df['approach'] = row['approach']
            detail_df['vlm_label'] = row['vlm_label']
            detail_df['llm_generator'] = row['llm_generator']
            detail_df['report'] = row['report']
            detail_df['testcase_file'] = testcase_file
            detail_df['description_file'] = desc_file
            objective_detail_tables.append(detail_df)

    except Exception as e:
        metric_errors.append({**row.to_dict(), 'metric_error': f'metric_error: {type(e).__name__}: {e}'})

metrics_df = pd.DataFrame(metric_rows)
metric_errors_df = pd.DataFrame(metric_errors)
objective_details_df = pd.concat(objective_detail_tables, ignore_index=True) if objective_detail_tables else pd.DataFrame()

print('Metric rows:', len(metrics_df))
print('Objective detail rows:', len(objective_details_df))
print('Metric errors:', len(metric_errors_df))

display(metrics_df.head())
display(metric_errors_df.head())

In [ ]:
# 13. Overall summaries

def weighted_coverage(sub):
    denom = sub['total_objectives'].sum()
    return sub['covered_objectives'].sum() / denom if denom else np.nan


def pct(x):
    return '' if pd.isna(x) else f'{x*100:.2f}%'


def make_group_summary(df, group_cols):
    summary = df.groupby(group_cols).agg(
        Rows=('report', 'count'),
        Reports=('report', 'nunique'),
        VLMs=('vlm_label', 'nunique'),
        LLMs=('llm_generator', 'nunique'),
        Generated_test_cases=('test_cases', 'sum'),
        Mean_test_cases_per_file=('test_cases', 'mean'),
        Mean_page_coverage=('page_coverage', 'mean'),
        Mean_component_actor_coverage=('component_or_actor_coverage', 'mean'),
        Mean_relationship_message_coverage=('relationship_or_message_coverage', 'mean'),
        Mean_main_flow_coverage=('main_flow_coverage', 'mean'),
        Mean_condition_branch_coverage=('condition_or_branch_coverage', 'mean'),
        Mean_overall_structural_coverage=('overall_structural_coverage', 'mean'),
        Total_objectives=('total_objectives', 'sum'),
        Covered_objectives=('covered_objectives', 'sum'),
        Total_exact_duplicates=('exact_duplicate_count', 'sum'),
        Mean_exact_duplicate_rate=('exact_duplicate_rate', 'mean'),
        Total_empty_required_fields=('total_empty_required_fields', 'sum'),
        Total_input_tokens=('input_tokens', 'sum'),
        Total_output_tokens=('output_tokens', 'sum'),
        Total_tokens=('total_tokens', 'sum'),
        Token_rows_found=('token_rows_found', 'sum'),
    ).reset_index()

    summary['Weighted_overall_structural_coverage'] = summary.apply(
        lambda r: r['Covered_objectives'] / r['Total_objectives'] if r['Total_objectives'] else np.nan,
        axis=1
    )
    return summary

# Main two-row table: direct vs two-stage.
main_summary_raw = make_group_summary(metrics_df, ['comparison_group', 'approach'])
main_table = main_summary_raw.copy()

# VLM-source summary: direct by VLM, two-stage averaged over LLMs for each VLM source.
vlm_summary_raw = make_group_summary(metrics_df, ['comparison_group', 'approach', 'vlm_label'])
vlm_table = vlm_summary_raw.copy()

# Full combination table: for appendix/debugging only.
combo_summary_raw = make_group_summary(metrics_df, ['comparison_group', 'approach', 'vlm_label', 'llm_generator'])
combo_table = combo_summary_raw.copy()

for df in [main_table, vlm_table, combo_table]:
    for c in [
        'Mean_page_coverage', 'Mean_component_actor_coverage', 'Mean_relationship_message_coverage',
        'Mean_main_flow_coverage', 'Mean_condition_branch_coverage', 'Mean_overall_structural_coverage',
        'Weighted_overall_structural_coverage', 'Mean_exact_duplicate_rate'
    ]:
        if c in df.columns:
            df[c] = df[c].apply(pct)
    if 'Mean_test_cases_per_file' in df.columns:
        df['Mean_test_cases_per_file'] = df['Mean_test_cases_per_file'].apply(lambda x: '' if pd.isna(x) else f'{x:.2f}')

print('MAIN TABLE:')
display(main_table)

print('VLM SOURCE SUMMARY:')
display(vlm_table)

print('FULL COMBINATION SUMMARY FOR APPENDIX/DEBUGGING:')
display(combo_table)